# Turnkey Release 1 Execution Pipeline (Kaggle Dual/Single Tesla T4)
### Project: *Dual-Stream Spatial-Frequency Feature Fusion with SNR-Adaptive Gating for Deepfake Detection: An Empirical Evaluation on Disjoint Partitions*
### Author: Yassin Yasser (Sadat Academy for Management Sciences)

This turnkey notebook executes the complete Release 1 pipeline:
1. **Dataset Integrity Verification**: Zero identity leakage check and verification of 0 DFD samples in train/val.
2. **Dual-Stream & Spatial Baseline Training**: 5 epochs on clean 37,104 train split (4-way balanced sampling, differential LRs `1e-5` / `1e-4`, FP16 AMP with FP32 FFT isolation).
3. **Macro-AUC Validation Checkpoint Selection**: Checkpoint selected based on balanced macro-AUC across FF++ and Celeb-DF Set A validation.
4. **Validation Calibration & Optimal Thresholding**: 4-way stratified Platt scaling, Youden's $J$ threshold $\tau^*$, and Bayesian dual thresholds $(\tau_{\mathrm{real}}, \tau_{\mathrm{fake}})$ with $\ge 98.0\%$ precision.
5. **Held-Out Test Evaluation (26,981 Crops)**:
   - Overall metrics, trivial baseline (always-fake), and Spatial-Only ConvNeXt baseline.
   - Per-source evaluation (FF++, Celeb-DF, DFD vs FF++, DFD vs YouTube) with 1,000-sample video-level clustered bootstrap 95% CIs.
   - Fine-grained subdomain breakdown.
6. **Spatiotemporal Bi-GRU Video Modeling**: 2-layer Bi-GRU with velocity deltas ($\Delta \mathbf{e}_t$) and dual-path attention/max-pooling.
7. **5-Fold Leave-One-Target-Out (LOTO)**: Generalization benchmark across unseen architectures.
8. **Robustness Stress-Testing**: Sweeps across JPEG, blur, noise, and downscaling.
9. **Single-T4 Latency Profiling**: Synchronized $B=1$ and $B=32$ throughput.
10. **Provenance & LaTeX Verification**: Writes `release1_results.json` and runs `verify_latex_full.py`.

In [ ]:
# 1. Environment & GPU Sanity Check
!nvidia-smi
%env NO_ALBUMENTATIONS_UPDATE=1
%env PYTHONUNBUFFERED=1

# Clone / Update Repository
%cd /kaggle/working
!if [ -d "deepfake-detection" ]; then cd deepfake-detection && git fetch origin && git reset --hard origin/main; else git clone https://github.com/yyouretoast/deepfake-detection.git; fi
%cd /kaggle/working/deepfake-detection
!pip install -q -r requirements.txt

In [ ]:
# 2. Locate Dataset and Verify Clean Release 1 Splits
import os, json, glob

candidates = [
    "/kaggle/input/deepfake-crops-512",
    "/kaggle/input/deepfake-crops-512/deepfake_crops_512",
    "/kaggle/input/deepfake-face-crops-256/deepfake_crops_512",
    "/kaggle/input/deepfake_crops_512",
    "./deepfake_crops_512"
]
data_dir = None
for c in candidates:
    if os.path.exists(os.path.join(c, "splits.json")):
        data_dir = os.path.abspath(c)
        break

print(f"Using Dataset Directory: {data_dir}")
with open(os.path.join(data_dir, "splits.json")) as f:
    s = json.load(f)
print(f"Split counts -> Train: {len(s['train']):,}, Val: {len(s['val']):,}, Test: {len(s['test']):,}")

In [ ]:
# 3. Execute Master Turnkey Pipeline Runner
# This single command runs all stages: training, macro-AUC selection, calibration,
# per-source evaluation with bootstrap CIs, spatiotemporal Bi-GRU, LOTO, robustness, and latency.
!python kaggle_pipeline/run_release1_kaggle.py \
    --data_dir {data_dir} \
    --output_dir /kaggle/working \
    --epochs 5 \
    --batch_size 32

In [ ]:
# 4. Display Formatted Release 1 Results Ledger
import json
results_path = "/kaggle/working/release1_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        res = json.load(f)
    print("="*60)
    print("MASTER RELEASE 1 RESULTS SUMMARY")
    print("="*60)
    print("Provenance:", json.dumps(res.get("provenance", {}), indent=2))
    print("\nCalibration:", json.dumps(res.get("calibration", {}), indent=2))
    print("\nOverall Test:", json.dumps(res.get("test_frame_evaluation", {}).get("overall", {}), indent=2))
    print("\nSpatial-Only Baseline:", json.dumps(res.get("test_frame_evaluation", {}).get("spatial_only", {}), indent=2))
    print("\nPer-Source Test:", json.dumps(res.get("test_frame_evaluation", {}).get("per_source", {}), indent=2))
    print("\nTemporal Bi-GRU:", json.dumps(res.get("temporal_video_evaluation", {}), indent=2))
    print("\nLOTO Folds:", json.dumps(res.get("loto_cross_generator_benchmark", {}), indent=2))
    print("\nLatency Profiling:", json.dumps(res.get("latency_profiling", {}), indent=2))
else:
    print(f"Results file not found at {results_path}")

In [ ]:
# 5. Synchronize LaTeX Manuscript Tables & Verify Against Live Run Results
!python scripts/generate_manuscript_tables.py --results /kaggle/working/release1_results.json --target manuscript/main.tex --update
!python scripts/verify_latex_full.py manuscript/main.tex --results /kaggle/working/release1_results.json
